# RISSK — scheduled pipeline runner (JupyterHub Notebook Jobs)

Each **configuration is a Kedro environment** (`conf/<config>/globals.yml`). This notebook runs
the pipeline **in-process** via `KedroSession` (plain Python, no subprocess): set `ENV` (which
configuration) and `PIPELINE` (which stage) below. There is no driver — the pipeline, storage,
and questionnaire all come from Kedro config.

**A configuration** (`conf/<config>/globals.yml`) sets:
- `survey` + a single `questionnaire` (`name`, `VERSION`, optional `filter_var`) — one questionnaire per env;
- three storage roots, whose *values* pick the storage mode:
  - `input_root` — where the export zips live: a local path, or `s3://<bucket>`.
  - `work_root` — always-local staging (zips are fetched + unzipped here; unzip is local-only).
  - `output_root` — where stages 20→40 land: a local path, or `s3://<bucket>` (written via `s3fs`, no aws CLI).

  → `local` / `s3in` / `s3out` / `s3` are just the four combinations of local vs `s3://` roots.

**`PIPELINE`** selects which stage to run: `"__default__"` (all of data_ingestion → feature_creation → rissk_scoring), or a single one of those.

Outputs are keyed by `<survey>`, so a survey folder holds **one** questionnaire's results.
**Several questionnaires = several envs**, each pointing at its own survey folder; set `ENV` to a list below to run them in one job.

**Prerequisites**

- Environment installed from the repo root: `conda env create -f environment.yml`
  (or `uv sync`). The single `rissk` package is installed editable.
- Kernel registered so Notebook Jobs can run on it (the kernel keeps the name `rissk_kedro`):
  `python -m ipykernel install --user --name rissk_kedro`
- For S3 roots: AWS credentials in the environment (standard chain — env vars or `~/.aws`; `s3fs` uses them).

The cell below is tagged `parameters`, so to schedule a different run you only override `ENV`
(and optionally `PIPELINE`) in the Notebook Jobs *Parameters* form (e.g. `ENV = "grdslchbs_test"`)
— one job per configuration, same notebook, no code changes.

In [ ]:
# Which configuration(s) to run — a Kedro env name under conf/<ENV>/, or a list of them.
# (Available envs are the conf/<name>/ folders, e.g. grdslchbs_test, s3in, s3out, s3.) 
# You can create your own env by copying one of the existing ones and modifying it.
ENV = "s3"

# Which pipeline to run: "__default__" (all stages) or one of
# "data_ingestion" / "feature_creation" / "rissk_scoring".
PIPELINE = "__default__"

In [ ]:
from pathlib import Path

import rissk
from kedro.framework.session import KedroSession
from kedro.framework.startup import bootstrap_project

# The Kedro project root (repo root), derived from the installed package.
PROJECT_ROOT = Path(rissk.__file__).resolve().parents[2]
bootstrap_project(PROJECT_ROOT)   # load project settings + pipelines once (env-independent)

envs = ENV if isinstance(ENV, (list, tuple)) else [ENV]

In [ ]:
# Run each env in its own KedroSession (in-process; env-specific globals + catalog).
# Failures are isolated so one bad configuration doesn't skip the rest of a multi-env
# run; the outcomes are collected and re-raised at the end so the job is marked failed.
results = {}
for env in envs:
    print(f"=== run --env {env} --pipeline {PIPELINE} ===", flush=True)
    try:
        with KedroSession.create(project_path=PROJECT_ROOT, env=env) as session:
            session.run(pipeline_names=[PIPELINE])   # "__default__" = all stages
        results[env] = "OK"
    except Exception as exc:
        results[env] = f"FAILED ({type(exc).__name__}: {exc})"
    print(f"--- {env}: {results[env]} ---", flush=True)

ok = sum(r == "OK" for r in results.values())
print(f"\nSummary: {ok}/{len(envs)} env(s) succeeded.", flush=True)
failed = [env for env, r in results.items() if r != "OK"]
if failed:
    raise RuntimeError(f"kedro run failed for env(s): {failed}")

### Where results land

Outputs are written under `output_root`, keyed by survey:

```
<output_root>/<survey>/latest/
    20_INTERIM/   ...
    30_PROCESSED/ ...
    41_SCORES/    item_scores.parquet, responsible_scores.csv, unit_rissk_scores.csv
```

- local roots → on disk; `s3://<bucket>` roots → written natively via `s3fs` (no upload step).
- input zips are always staged + unzipped under the local `work_root` first (unzip is local-only).
- a survey folder holds **one** questionnaire's results — for several questionnaires, use several envs pointing at separate survey folders.

The final scores file is `41_SCORES/unit_rissk_scores.csv` — point downstream consumers there.